# python_on_whales smoke test

Pure-Python Docker client wrapper. We test imports + offline API surface (CLI builders, type construction, parsing) without requiring a running Docker daemon.

In [ ]:
# Example 1: Public API surface — DockerClient and core classes are importable.
from python_on_whales import DockerClient, docker
from python_on_whales import Container, Image, Volume, Network

# The top-level `docker` object is a default DockerClient instance.
assert isinstance(docker, DockerClient)

# Core entity classes exist and are types.
for cls in (Container, Image, Volume, Network):
    assert isinstance(cls, type), f"{cls.__name__} is not a class"
    print(f"  {cls.__module__}.{cls.__name__}")

print(f"\ndocker = {docker!r}")

In [ ]:
# Example 2: DockerClient builds its CLI command; `env` stands in for the docker binary
# so this runs without Docker installed.
import shutil

from python_on_whales import DockerClient

client = DockerClient(
    host="ssh://user@example.com",
    debug=True,
    client_call=["env"],
)

# client_config holds the assembled CLI prefix used for every invocation.
cfg = client.client_config
print(f"host          = {cfg.host}")
print(f"debug         = {cfg.debug}")
print(f"client_call   = {cfg.client_call}")

# The assembled command resolves the binary path + appends --debug/--host.
docker_cmd = cfg.docker_cmd
print(f"docker_cmd    = {docker_cmd}")
assert str(docker_cmd[0]) == shutil.which("env")
assert "--debug" in [str(part) for part in docker_cmd]
assert any("ssh://user@example.com" in str(part) for part in docker_cmd)

In [ ]:
# Example 3: Sub-command namespaces — DockerClient exposes the full CLI surface.
from python_on_whales import DockerClient

client = DockerClient()

# Every major Docker CLI noun is reachable as an attribute on the client.
namespaces = [
    "container", "image", "volume", "network", "compose",
    "buildx", "system", "swarm", "service", "node",
    "context", "manifest", "plugin", "stack", "secret",
]
for ns in namespaces:
    obj = getattr(client, ns)
    assert obj is not None, f"missing namespace {ns}"
    print(f"  client.{ns:<10} -> {type(obj).__name__}")

In [ ]:
# Example 4: Exception hierarchy — error classes are wired up.
from python_on_whales.exceptions import (
    DockerException,
    NoSuchContainer,
    NoSuchImage,
    NoSuchVolume,
    NoSuchNetwork,
)

# All specific errors should subclass the base DockerException.
for exc in (NoSuchContainer, NoSuchImage, NoSuchVolume, NoSuchNetwork):
    assert issubclass(exc, DockerException), f"{exc.__name__} not a DockerException"
    print(f"  {exc.__name__} <- DockerException")

# Constructing a DockerException requires command_launched + return_code; the
# captured stdout/stderr are the raw subprocess BYTES (decoded internally).
try:
    raise DockerException(["docker", "ps"], 1, stderr=b"boom")
except DockerException as e:
    caught = e

assert caught.return_code == 1
assert caught.stderr == "boom"
print(f"\ncaught: {type(caught).__name__}: rc={caught.return_code}")

In [ ]:
# Example 5: Trying to call docker without a running daemon raises cleanly.
import shutil

from python_on_whales import DockerClient
from python_on_whales.client_config import ClientNotFoundError
from python_on_whales.exceptions import DockerException

# Point at a binary that definitely does not exist so we exercise the error path
# without depending on whether docker is installed in the test environment.
# A missing binary raises ClientNotFoundError (subclass of Exception, NOT of
# DockerException) from the CLI path resolution.
missing = "docker-does-not-exist-zzz"
assert shutil.which(missing) is None

client = DockerClient(client_call=[missing])
raised = False
try:
    client.ps()
except (DockerException, ClientNotFoundError, FileNotFoundError, OSError) as e:
    raised = True
    print(f"got expected error: {type(e).__name__}: {str(e)[:120]}")

assert raised, "expected ClientNotFoundError / DockerException when calling a missing CLI"
print("\npython_on_whales smoke test PASSED")